# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [4]:
openai = OpenAI()

In [5]:
# Some lists!

todos = []
completed = []

In [6]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [ ]:
get_todo_report()

In [8]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [9]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [10]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [13]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [14]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [15]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [18]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [19]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [20]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [21]:
todos, completed = [], []
loop(messages)

Todo #1: Extract given data and define variables (distance between Boston and New York).
Todo #2: Estimate missing quantity: approximate Boston–New York distance.
Todo #3: Set up relative motion equation with a 1-hour head start for Boston train.
Todo #4: Solve for meeting time and convert to clock time.
Todo #5: Sanity-check result and present final answer clearly.

Given:
- Train B (from Boston): departs 2:00 pm at 60 mph.
- Train N (from New York): departs 3:00 pm at 80 mph toward Boston.

Let D = distance between Boston and New York (miles).
Let t = hours after 3:00 pm until they meet.

Todo #1: Extract given data and define variables (distance between Boston and New York).
Todo #2: Estimate missing quantity: approximate Boston–New York distance.
Todo #3: Set up relative motion equation with a 1-hour head start for Boston train.
Todo #4: Solve for meeting time and convert to clock time.
Todo #5: Sanity-check result and present final answer clearly.

The problem does not state D. Use a reasonable real-world estimate.

A common road distance BostoncNew York City is about 215 miles; rail is similar order. Use D \approx 215 miles as a
reasonable estimate.

Todo #1: Extract given data and define variables (distance between Boston and New York).
Todo #2: Estimate missing quantity: approximate Boston–New York distance.
Todo #3: Set up relative motion equation with a 1-hour head start for Boston train.
Todo #4: Solve for meeting time and convert to clock time.
Todo #5: Sanity-check result and present final answer clearly.

...

Solve: t = (D - 60)/140.
With D \approx 215: t \approx (215 - 60)/140 = 155/140 \approx 1.1071 h.

Convert 0.1071 h to minutes: 0.1071\times 60 \approx 6.43 minutes.
So meeting time \approx 3:00 pm + 1 h 6.4 min = 4:06 pm (approx).

Todo #1: Extract given data and define variables (distance between Boston and New York).
Todo #2: Estimate missing quantity: approximate Boston–New York distance.
Todo #3: Set up relative motion equation with a 1-hour head start for Boston train.
Todo #4: Solve for meeting time and convert to clock time.
Todo #5: Sanity-check result and present final answer clearly.

Check distances at ~4:06 pm:
- Train B travel time since 2:00 pm: 2.1071 h, distance \approx 60\times2.1071=126.4 mi.
- Train N travel time since 3:00 pm: 1.1071 h, distance \approx 80\times1.1071=88.6 mi.
Total \approx 215.0 mi, consistent with assumed D.

Also provide formula in terms of D: meeting time is 3:00 pm + (D-60)/140 hours.

Todo #1: Extract given data and define variables (distance between Boston and New York).
Todo #2: Estimate missing quantity: approximate Boston–New York distance.
Todo #3: Set up relative motion equation with a 1-hour head start for Boston train.
Todo #4: Solve for meeting time and convert to clock time.
Todo #5: Sanity-check result and present final answer clearly.

Assuming a reasonable Boston–New York distance of about 215 miles:

- By 3:00 pm, the Boston train has gone 60 miles (1 hour × 60 mph), leaving 215 − 60 = 155 miles between them.
- From 3:00 pm on, they approach each other at a combined speed of 60 + 80 = 140 mph.
- Time after 3:00 pm to meet:
  t = 155 / 140 ≈ 1.107 hours ≈ 1 hour 6 minutes

So they meet at approximately 4:06 pm.

In general, if the distance is D miles, they meet at 3:00 pm + (D − 60)/140 hours.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>